In [18]:
## Vloco de codigo para instalar versao especifica dos pacotes
##pip install pandas==1.5.3
##pip install numpy==1.24.3

In [19]:
import sys
import os

# Acesso aos módulos do diretório
from pathlib import Path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))
print("Project root:", project_root)

Project root: C:\pod\hackathon_pod_2025


##### Carregando pacotes

In [20]:
# Pacotes de manipulacao

import pandas as pd
import numpy as np
import os

# Pacotes de visualizacao
import matplotlib.pyplot as plt
import seaborn as sns

# Funcoes customizadas
import configs.function_basic as funcoes

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

pandas: 1.5.3
numpy: 1.26.4


## Carregando databases

#### Book_02

In [21]:
# Carregando book_02
book_02 = pd.read_parquet(project_root/'database/processed/book_variaveis_02.parquet')
print("Book 02 data shape:", book_02.shape)

Book 02 data shape: (1280828, 30)


In [22]:
book_02.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1280828 entries, 0 to 1290525
Data columns (total 30 columns):
 #   Column             Non-Null Count    Dtype         
---  ------             --------------    -----         
 0   SAFRA              1280828 non-null  int64         
 1   FPD                1280828 non-null  int64         
 2   SCORE_01           1273364 non-null  float64       
 3   SCORE_02           1280256 non-null  float64       
 4   NUM_CPF            1280828 non-null  object        
 5   SCORE_RATEO        1272792 non-null  float64       
 6   SCORE_AVG          1272792 non-null  float64       
 7   SCORE_DIFF         1272792 non-null  float64       
 8   SCORE_MIN          1280828 non-null  float64       
 9   DATADENASCIMENTO   1280828 non-null  datetime64[ns]
 10  var_03             1198460 non-null  object        
 11  var_04             1280828 non-null  object        
 12  var_05             1228039 non-null  object        
 13  var_09             600120 n

#### Base Dados Telco

In [23]:
## Carregando todos arquivos em parquet de uma pasta

all_files = [os.path.join(project_root/'database/raw/base_telco/base_telco/', f) for f in os.listdir(project_root/'database/raw/base_telco/base_telco/') if f.endswith('.parquet')]
df_list = [pd.read_parquet(f, engine='pyarrow') for f in all_files]

df_dados_telco = pd.concat(df_list, ignore_index=True)
print('Base Dados Telco data shape:', df_dados_telco.shape)

Base Dados Telco data shape: (1367104, 74)


In [24]:
df_dados_telco.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1367104 entries, 0 to 1367103
Data columns (total 74 columns):
 #   Column           Non-Null Count    Dtype 
---  ------           --------------    ----- 
 0   NUM_CPF          1367104 non-null  object
 1   SAFRA            1367104 non-null  object
 2   FLAG_INSTALACAO  1367104 non-null  object
 3   FPD              1321168 non-null  object
 4   PROD             1367104 non-null  object
 5   flag_mig2        1308974 non-null  object
 6   var_26           1365809 non-null  object
 7   var_27           1365809 non-null  object
 8   var_28           1365809 non-null  object
 9   var_29           1365809 non-null  object
 10  var_30           1365809 non-null  object
 11  var_31           1365809 non-null  object
 12  var_32           1365809 non-null  object
 13  var_33           1365809 non-null  object
 14  var_34           1365809 non-null  object
 15  var_35           1365809 non-null  object
 16  var_36           1365809 non-null  o

Como na analise exploratoria vimos que estes são todos valores numericos continuos, iremos transformar todas as `var_x` em numericas.

Iremos ajustar aqui pois o book_02 possui este mesmo padrao de colunas mas nao sao numericas.

In [25]:
funcoes.convert_var_columns_to_numeric(df_dados_telco, inplace=True)

#### Merge dos Datasets

In [26]:
# Primeiro vamos transformar a coluna SAFRA para o mesmo formato de book_02
df_dados_telco['SAFRA'] = df_dados_telco['SAFRA'].astype('int64')

In [27]:
cols_to_drop = [
    col for col in df_dados_telco.columns
    if col in book_02.columns
    and col not in ['SAFRA', 'NUM_CPF']
]

df_dados_telco_clean = df_dados_telco.drop(columns=cols_to_drop)

df_book_03 = pd.merge(
    book_02,
    df_dados_telco_clean,
    how='left',
    on=['SAFRA', 'NUM_CPF']
)

In [28]:
# Sanity check
book_02.shape[0] == df_book_03.shape[0]

True

In [29]:
funcoes.generate_metadata(df_book_03)

,nome_variavel,tipo,qt_nulos,percent_nulos,cardinalidade
0,var_09,object,680708,53.15,15
1,TEMPO_CADASTRO,Int64,492911,38.48,62
2,var_25,object,131241,10.25,61
3,var_03,object,82368,6.43,100
4,CEP_3_digitos,object,74605,5.82,901
...,...,...,...,...,...
96,SUB_REGIAO_POSTAL,object,0,0.00,101
97,REGIAO_POSTAL_TXT,object,0,0.00,11
98,FLAG_INSTALACAO,object,0,0.00,1
99,PROD,object,0,0.00,1


### Feature Engineer


##### Bloco 01

Deletaremos as colunas que possuem cardinalidade igual a 1

Como este dataset possui muitas colunas e o trabalho de entender uma por uma é exaustivo, primeiro iremos verificar se temos colunas com possuem exatamente o mesmo valor para cada linha

#### Bloco 01

In [30]:
funcoes.drop_single_cardinality_columns(df_book_03)

🧹 Colunas removidas (cardinalidade = 1): 3
 - FLAG_INSTALACAO
 - PROD
 - flag_mig2


,SAFRA,FPD,SCORE_01,SCORE_02,NUM_CPF,SCORE_RATEO,SCORE_AVG,SCORE_DIFF,SCORE_MIN,DATADENASCIMENTO,...,var_84,var_85,var_86,var_87,var_88,var_89,var_90,var_91,var_92,var_93
0,202410,0,562.0,636.0,ZZZZZX7XWY8,1.131673,599.0,74.0,562.0,1983-12-26,...,304,304,2,3,1,50,0.86,2,1,1
1,202410,1,546.0,518.0,ZZZZZX88YXY,0.948718,532.0,-28.0,518.0,1980-12-24,...,304,304,304,304,304,304,304.00,304,304,304
2,202410,0,621.0,750.0,ZZZZZYT7XYT,1.207729,685.5,129.0,621.0,1982-07-23,...,304,304,0,1,1,100,298.00,1,1,2
3,202410,1,609.0,679.0,ZZZZZNTXY9Z,1.114943,644.0,70.0,609.0,1988-09-27,...,1,2,304,304,304,304,304.00,304,304,304
4,202410,0,621.0,722.0,ZZZZZ79ZXUX,1.162641,671.5,101.0,621.0,1984-08-04,...,304,304,304,304,304,304,304.00,304,304,304
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1280823,202503,0,604.0,674.0,99997YWXNZZ,1.115894,639.0,70.0,604.0,1960-01-05,...,304,304,304,304,304,304,304.00,304,304,304
1280824,202503,0,688.0,765.0,99998TYXZN8,1.111919,726.5,77.0,688.0,1982-12-25,...,304,304,304,304,304,304,304.00,304,304,304
1280825,202503,0,616.0,630.0,9999888YYU9,1.022727,623.0,14.0,616.0,1988-12-26,...,304,304,4,3,6,0,1.56,6,4,0
1280826,202503,0,627.0,649.0,9999889ZN9X,1.035088,638.0,22.0,627.0,1963-09-15,...,304,304,3,2,1,0,1.16,5,1,0


In [31]:
# Conferindo colunas que possuem o mesmo valor
colunas_duplicadas = funcoes.find_duplicate_columns(df_book_03)
print("Colunas com valores duplicados:", colunas_duplicadas)

Colunas com valores duplicados: []


#### Ajustando os tipos de dados

Durante o processo inteiro os tipos de dados foram criadas da forma correta.

Não iremos realizar mais nenhum tratamento destes dados, pois são todos numericos continuos. A partir daqui iremos revisar apenas com a solicitação do time de Ciencia de Dados, caso alguma variável se mostre interessante.

In [32]:
# Criando novo dataset
book_variaveis_03 = df_book_03.copy()

In [33]:
# Salvando o dataframe em csv
book_variaveis_03.to_parquet(project_root/'database/processed/book_variaveis_03.parquet', index=False)